In [ ]:
from transformers import AutoTokenizer, AutoProcessor, TokenizersBackend
from qwen_vl_utils import process_vision_info

# Load model

In [50]:
model_path = "models/Qwen3-VL-2B-Instruct"

In [51]:
tokenizer: TokenizersBackend = AutoTokenizer.from_pretrained(model_path)
tokenizer

Qwen2Tokenizer(name_or_path='models/Qwen3-VL-2B-Instruct', vocab_size=151643, model_max_length=262144, padding_side='right', truncation_side='right', special_tokens={'eos_token': '<|im_end|>', 'pad_token': '<|endoftext|>'}, added_tokens_decoder={
	151643: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151644: AddedToken("<|im_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151645: AddedToken("<|im_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151646: AddedToken("<|object_ref_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151647: AddedToken("<|object_ref_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151648: AddedToken("<|box_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151649: AddedToken("<|box_end|>", rstrip=F

In [ ]:
processor = AutoProcessor.from_pretrained(model_path)
processor

Qwen3VLProcessor:
- image_processor: Qwen2VLImageProcessor {
  "do_convert_rgb": true,
  "do_normalize": true,
  "do_rescale": true,
  "do_resize": true,
  "image_mean": [
    0.5,
    0.5,
    0.5
  ],
  "image_processor_type": "Qwen2VLImageProcessor",
  "image_std": [
    0.5,
    0.5,
    0.5
  ],
  "merge_size": 2,
  "patch_size": 16,
  "resample": 3,
  "rescale_factor": 0.00392156862745098,
  "size": {
    "longest_edge": 16777216,
    "shortest_edge": 65536
  },
  "temporal_patch_size": 2
}

- tokenizer: Qwen2Tokenizer(name_or_path='models/Qwen3-VL-2B-Instruct', vocab_size=151643, model_max_length=262144, padding_side='right', truncation_side='right', special_tokens={'eos_token': '<|im_end|>', 'pad_token': '<|endoftext|>'}, added_tokens_decoder={
	151643: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151644: AddedToken("<|im_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151

In [23]:
[i for i in dir(processor) if not i.startswith('__')]

['_auto_class',
 '_calculate_timestamps',
 '_check_special_mm_tokens',
 '_get_arguments_from_pretrained',
 '_get_files_timestamps',
 '_get_num_multimodal_tokens',
 '_load_tokenizer_from_pretrained',
 '_merge_kwargs',
 '_upload_modified_files',
 'apply_chat_template',
 'audio_ids',
 'batch_decode',
 'chat_template',
 'check_argument_for_proper_class',
 'create_mm_token_type_ids',
 'decode',
 'from_args_and_dict',
 'from_pretrained',
 'get_attributes',
 'get_possibly_dynamic_module',
 'get_processor_dict',
 'image_ids',
 'image_processor',
 'image_token',
 'image_token_id',
 'model_input_names',
 'parse_response',
 'post_process_image_text_to_text',
 'post_process_multimodal_output',
 'push_to_hub',
 'register_for_auto_class',
 'save_pretrained',
 'to_dict',
 'to_json_file',
 'to_json_string',
 'tokenizer',
 'valid_processor_kwargs',
 'validate_init_kwargs',
 'video_ids',
 'video_processor',
 'video_token',
 'video_token_id',
 'vision_end_token',
 'vision_end_token_id',
 'vision_start_to

# apply_chat_template

In [37]:
conversation = [
    {
        "role": "user",
        "content": [
            {
                "type": "image",
                "image": "./images/image1.jpg",
            },
            {"type": "text", "text": "描述一下上面图片的内容"},
        ],
    }
]

In [43]:
inputs = tokenizer.apply_chat_template(
    conversation=conversation,
    add_generation_prompt=False,
    tokenize=True,
    padding=False,
    truncation=False,
    max_length=None,
    return_tensors="pt",
)
inputs

{'input_ids': tensor([[151644,    872,    198, 151652, 151655, 151653,  53481, 100158, 104135,
          45930, 104597, 151645,    198]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

In [44]:
inputs["input_ids"].shape

torch.Size([1, 13])

In [38]:
# same as processor.apply_chat_template
text_with_template = tokenizer.apply_chat_template(
    conversation=conversation,
    add_generation_prompt=False,
    tokenize=False,
    padding=False,
    truncation=False,
    max_length=None,
    return_tensors="pt",
)
text_with_template

'<|im_start|>user\n<|vision_start|><|image_pad|><|vision_end|>描述一下上面图片的内容<|im_end|>\n'

# 提取视觉信息

In [ ]:
# 这是一个官方提供的辅助函数，用于从 messages 中提取并预处理图像/视频流
image_inputs, video_inputs = process_vision_info(conversation)
image_inputs

[<PIL.Image.Image image mode=RGB size=1288x1288>]

# processor

In [45]:
inputs = processor(
    text=text_with_template,  # 可以为列表或单个string
    images=image_inputs,  # 可以为列表或单张图片
    videos=video_inputs,
    return_tensors="pt",  # 返回数据格式 np pt tf jax
)
inputs

{'input_ids': tensor([[151644,    872,    198,  ..., 104597, 151645,    198]]), 'attention_mask': tensor([[1, 1, 1,  ..., 1, 1, 1]]), 'mm_token_type_ids': tensor([[0, 0, 0,  ..., 0, 0, 0]]), 'pixel_values': tensor([[1., 1., 1.,  ..., 1., 1., 1.],
        [1., 1., 1.,  ..., 1., 1., 1.],
        [1., 1., 1.,  ..., 1., 1., 1.],
        ...,
        [1., 1., 1.,  ..., 1., 1., 1.],
        [1., 1., 1.,  ..., 1., 1., 1.],
        [1., 1., 1.,  ..., 1., 1., 1.]]), 'image_grid_thw': tensor([[ 1, 80, 80]])}

In [46]:
inputs["input_ids"].shape

torch.Size([1, 1612])

In [ ]:
inputs["image_grid_thw"]
# T (Time): 时间维度。这里是 1，因为输入的是单张静态图片；如果是视频，这里就是采样的帧数。
# H (Height): 垂直方向上的 Patch 数量。这里是 80。
# W (Width): 水平方向上的 Patch 数量。这里是 80。

In [ ]:
# 图像分辨率为 1288 x 1288
# vit 的 patch_size 为 16
# 因此图片分辨率缩放到 16 最近的的倍数为 1280
# 1280 / 16 * 1280 / 16 = 80 * 80 = 6400
# patch 数量为 6400
# merge_size 为 2
# 因此最终 token 数量为 6400 / 2 / 2 = 1600
# 上面 apply_chat_template 直接转换 token 时数量为 13
# 计算 token 数量, 要减去 <|image_pad|> 占位符, 1600 + 13 - 1 = 1612


tensor([[ 1, 80, 80]])